# 🚀 ShardFlow — Pipeline Node (Colab GPU Node)

Run this cell on Google Colab (T4 / A100 GPU) to join the ShardFlow distributed pipeline cluster.

In [ ]:
# Step 1: Install ShardFlow dependencies & clone repository
!pip install -q torch transformers tokenizers safetensors accelerate requests pydantic sse-starlette
!git clone https://github.com/adityaraut/Shardflow.git /content/Shardflow 2>/dev/null || (cd /content/Shardflow && git pull)
%cd /content/Shardflow
!pip install -q -e .

In [ ]:
# Step 2: Launch Colab Node & Connect to Render Gateway
import asyncio, os, sys, time, requests, torch
from shardflow.node.layer_loader import load_layer_slice
from shardflow.node.node import PipelineNode
from shardflow.transport.tunnel import start_bore_tunnel

# CONFIGURATION
REGISTRY_URL = "https://shardflow.onrender.com"
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # Or Qwen/Qwen2.5-7B-Instruct
NODE_ID = f"colab-node-{int(time.time()) % 10000}"
LOCAL_PORT = 9000

# 1. Start TCP Tunnel
proc, public_host, public_port = start_bore_tunnel(LOCAL_PORT)
print(f"📡 Public TCP Tunnel established: {public_host}:{public_port}")

# 2. Register node with Render Topology Registry
vram = torch.cuda.get_device_properties(0).total_memory / (1024 * 1024) if torch.cuda.is_available() else 0.0
reg_payload = {
    "node_id": NODE_ID,
    "addr": public_host,
    "port": public_port,
    "vram_available_mb": vram,
    "vram_total_mb": vram,
    "model_id": MODEL_ID,
}
reg_resp = requests.post(f"{REGISTRY_URL}/register", json=reg_payload, timeout=10.0).json()
print(f"✅ Registered {NODE_ID} -> Assigned Layers [{reg_resp['layer_start']}, {reg_resp['layer_end']})")

# 3. Load assigned model layer slice into GPU memory
model_slice = load_layer_slice(
    model_path=MODEL_ID,
    layer_start=reg_resp["layer_start"],
    layer_end=reg_resp["layer_end"],
    include_norm=reg_resp["is_last_node"],
    include_lm_head=reg_resp["is_last_node"],
    device="cuda" if torch.cuda.is_available() else "cpu",
)

# 4. Initialize and run Pipeline Node server
node = PipelineNode(
    model_slice=model_slice,
    is_first_node=reg_resp["is_first_node"],
    is_last_node=reg_resp["is_last_node"],
    next_node_host=reg_resp.get("next_node_host"),
    next_node_port=reg_resp.get("next_node_port"),
    listen_host="0.0.0.0",
    listen_port=LOCAL_PORT,
)

print(f"🚀 Node active & listening on port {LOCAL_PORT}!")
await node.serve_forever()